# QwerySmith - Train (Colab T4)

Trains the QLoRA behaviour tune on a dataset's RAFT triples. **Runtime -> Change runtime type -> T4 GPU.**

Everything before this step (ingest, validate, retrieve, triples) runs on CPU and is re-run here from raw data so the Colab run is self-contained and reproducible.

- `DATASET` selects the dataset dir. `online_retail_ii` downloads its public UCI data automatically (no credentials). `olist` needs the Kaggle CSVs (put them in Drive; the cell copies them).
- Set the `HF_TOKEN` Colab secret (key icon) to publish.

| Cell | Purpose |
|---|---|
| 1 | Setup: deps, GPU, repo clone, raw data |
| 2 | Prepare: ingest -> profile -> validate -> retrieve -> triples (CPU) |
| 3 | Train all seeds (QLoRA) |
| 4 | Contract sanity check |
| 5 | Publish to Hugging Face |
| 6 | Save a zip to Drive |

In [ ]:
# ===== Cell 1: setup =====
DATASET = "online_retail_ii"   # or "olist"
REPO_URL = "https://github.com/Cyrax321/QwerySmith-1.0.git"
REPO = "/content/qwerysmith"

# install EVERYTHING up front, before anything imports torch in this kernel.
# (importing torch before the ML install can pin a stale build and break unsloth)
%pip install -q typer pyyaml pydantic sqlglot sqlalchemy unsloth trl datasets peft transformers accelerate bitsandbytes

import os, subprocess, shutil, json

# GPU check via subprocess so the kernel stays clean
check = subprocess.run(["python", "-c",
    "import torch; assert torch.cuda.is_available(), 'NO CUDA'; "
    "print(torch.cuda.get_device_name(0), torch.cuda.get_device_properties(0).total_memory//2**20)"],
    capture_output=True, text=True)
print(check.stdout.strip() or check.stderr.strip())
assert "NO CUDA" not in check.stderr, "Runtime -> Change runtime type -> T4 GPU"

if not os.path.exists(REPO):
    subprocess.run(["git", "clone", "-q", REPO_URL, REPO], check=True)
else:
    subprocess.run(["git", "-C", REPO, "pull", "-q"], check=True)
os.chdir(REPO)

raw_dir = f"{REPO}/datasets/{DATASET}/raw"
os.makedirs(raw_dir, exist_ok=True)

if DATASET == "online_retail_ii" and len([f for f in os.listdir(raw_dir) if f.endswith(".csv")]) < 2:
    # public UCI download, no credentials
    subprocess.run("curl -sL 'https://archive.ics.uci.edu/static/public/502/online+retail+ii.zip' -o /tmp/retail.zip", shell=True, check=True)
    subprocess.run("unzip -o -q /tmp/retail.zip -d /tmp/uci", shell=True, check=True)
    subprocess.run(["pip", "install", "-q", "openpyxl"], check=True)
    import openpyxl, csv
    wb = openpyxl.load_workbook("/tmp/uci/online_retail_II.xlsx", read_only=True)
    for sheet in wb.sheetnames:
        with open(f"{raw_dir}/{sheet}.csv", "w", newline="", encoding="utf-8") as f:
            w = csv.writer(f)
            for row in wb[sheet].iter_rows(values_only=True):
                w.writerow(["" if v is None else v for v in row])
        print("wrote", sheet)

if DATASET == "olist":
    if len([f for f in os.listdir(raw_dir) if f.endswith(".csv")]) < 9:
        # public Kaggle dataset: kagglehub fetches it anonymously (verified);
        # Drive is the fallback if Kaggle ever asks for auth on Colab
        try:
            subprocess.run(["pip", "install", "-q", "kagglehub"], check=True)
            import kagglehub
            kpath = kagglehub.dataset_download("olistbr/brazilian-ecommerce")
            for f in os.listdir(kpath):
                if f.endswith(".csv"):
                    shutil.copy(os.path.join(kpath, f), f"{raw_dir}/{f}")
            print("downloaded via kagglehub:", kpath)
        except Exception as e:
            print("kagglehub failed:", e)
            print("fallback: mount Drive with the 9 CSVs in qwerysmith_v3/raw")
            from google.colab import drive
            drive.mount("/content/drive")
            src = "/content/drive/MyDrive/qwerysmith_v3/raw"
            for f in os.listdir(src):
                if f.endswith(".csv"):
                    shutil.copy(f"{src}/{f}", f"{raw_dir}/{f}")
    n_olist = len([f for f in os.listdir(raw_dir) if f.endswith(".csv")])
    assert n_olist == 9, f"need all 9 Olist CSVs, found {n_olist}"

assert os.path.exists(f"{REPO}/datasets/{DATASET}/questions_v1.jsonl"), (
    "questions_v1.jsonl missing - commit the reviewed question set first")
print("raw files:", [f for f in os.listdir(raw_dir) if f.endswith(".csv")])
print("setup OK for", DATASET)

In [ ]:
# ===== Cell 2: prepare (CPU) =====
for stage in ["ingest", "profile", "validate", "retrieve", "triples"]:
    print(f"=== {stage} ===")
    r = subprocess.run(["python", "-m", "qwery_smith", stage, DATASET],
                       capture_output=True, text=True)
    print(r.stdout[-1500:])
    if r.returncode != 0:
        print(r.stderr[-1500:])
        raise SystemExit(f"{stage} FAILED")
# validate must pass: scorable denominator + holdout-leak checks green

In [ ]:
# ===== Cell 3: train all seeds =====
# unsloth and friends were installed in Cell 1; go straight to training

r = subprocess.run(["python", "-m", "qwery_smith", "train", DATASET,
                    "--seeds", "1,2,3", "--execute"],
                   capture_output=True, text=True)
print(r.stdout[-4000:])
if r.returncode != 0:
    print(r.stderr[-4000:])
    raise SystemExit("training FAILED")

In [ ]:
# ===== Cell 4: contract sanity check (seed 1) =====
import json, yaml, pathlib
from unsloth import FastLanguageModel

run_dir = sorted(pathlib.Path(f"{REPO}/runs/{DATASET}").glob("train_*"))[-1]
cfg = yaml.safe_load(open(run_dir / "qlora_seed1.yaml"))

# unsloth detects adapter_config.json in the dir and loads base + LoRA together
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=str(run_dir / "adapters" / "adapter_seed1"),
    max_seq_length=cfg["batch"]["max_len"],
    load_in_4bit=True,
)
FastLanguageModel.for_inference(model)

prepared = pathlib.Path(f"{REPO}/datasets/{DATASET}/prepared")
triples_path = sorted(prepared.glob("triples_seed*.jsonl"))[-1]   # any distractor seed
triple = json.loads(open(triples_path).readline())
inputs = tokenizer(triple["prompt"] + "\n\nASSISTANT:\n", return_tensors="pt").to("cuda")
out = model.generate(**inputs, max_new_tokens=512, temperature=0.7, top_p=0.8, do_sample=True)
text = tokenizer.decode(out[0][inputs.input_ids.shape[-1]:], skip_special_tokens=True)
print(text[:600])
assert "SQL:" in text and ("ANSWER:" in text or "REFUSAL:" in text), "adapter violates output contract"
print("contract OK")

In [ ]:
# ===== Cell 5: publish to Hugging Face =====
# Validation runs: use a dev prefix so the official release name stays clean
MODEL_PREFIX = "QwerySmith-2.0-dev"   # official release: "QwerySmith-2.0"
from google.colab import userdata
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

r = subprocess.run(["python", "-m", "qwery_smith", "publish", DATASET,
                    "--hf-user", "Cyrax321", "--model-prefix", MODEL_PREFIX],
                   capture_output=True, text=True)
print(r.stdout[-3000:])
if r.returncode != 0:
    print(r.stderr[-3000:])
    print("(fix token/permissions, or drop this cell)")

In [ ]:
# ===== Cell 6: zip artifacts to Drive =====
import pathlib, subprocess, os, shutil
from google.colab import drive
drive.mount('/content/drive')   # unconditional - without a mount the target is a LOCAL dir

DATA = '/content/drive/MyDrive/qwerysmith_v3'
os.makedirs(f"{DATA}/raw", exist_ok=True)

# raw CSVs too - the eval notebook rebuilds the identical DB from these
for f in os.listdir(raw_dir):
    if f.endswith(".csv"):
        shutil.copy(f"{raw_dir}/{f}", f"{DATA}/raw/{f}")

run_dir = sorted(pathlib.Path(f"{REPO}/runs/{DATASET}").glob("train_*"))[-1]
target = f"{DATA}/run_{DATASET}_{run_dir.name}.zip"
subprocess.run(["zip", "-qr", target, f"runs/{DATASET}/{run_dir.name}",
                f"datasets/{DATASET}/prepared"], cwd=REPO, check=True)
print("zipped ->", target)
print("adapters:", [p.name for p in (run_dir / "adapters").glob("adapter_seed*")])